In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

# Настройка путей и параметров
TRAIN_PATH = 'train_2.csv'
BASELINE_SUB_PATH = 'test_baseline.csv'  # Сюда должен сохранять результат pipeline.py
FINAL_SUB_PATH = 'test.csv'              # Итоговый скорректированный прогноз
APE_THRESHOLD = 100.0                    # Порог аномальности в процентах (APE)


def load_history(path):
    """Загрузка исторических данных и расчет базового индекса времени."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл с историей не найден по пути: {path}")
    
    df = pd.read_csv(path)
    df['time_idx'] = df['Год'] * 12 + df['Месяц']
    return df.sort_values(['new_id', 'time_idx']).reset_index(drop=True)


def calculate_fallback_predictions(df):
    """
    Расчет робастных исторических прогнозов на март 2025 года.
    Используется комбинация 70% SNaive и 30% трехмесячного скользящего среднего.
    """
    pivot_df = df.pivot(index='new_id', columns='time_idx', values='РТО')

    # Индексы временных периодов относительно марта 2025
    idx_jan_25 = 2025 * 12 + 1   # lag_2
    idx_dec_24 = 2024 * 12 + 12  # lag_3
    idx_nov_24 = 2024 * 12 + 11  # lag_4
    idx_mar_24 = 2024 * 12 + 3   # lag_12
    idx_jan_24 = 2024 * 12 + 1   # lag_14

    rto_jan_25 = pivot_df[idx_jan_25]
    rto_dec_24 = pivot_df[idx_dec_24]
    rto_nov_24 = pivot_df[idx_nov_24]
    rto_mar_24 = pivot_df[idx_mar_24]
    rto_jan_24 = pivot_df[idx_jan_24]
    rto_mean = pivot_df.mean(axis=1)

    # Математические проекции тренда без привлечения ML-моделей
    robust_snaive = rto_mar_24 * (rto_jan_25 / (rto_jan_24 + 1e-6))
    robust_rolling = (rto_jan_25 + rto_dec_24 + rto_nov_24) / 3.0

    fallback_preds = 0.7 * robust_snaive + 0.3 * robust_rolling
    
    # Резервное заполнение возможных NaN-значений средними показателями магазина
    fallback_preds = fallback_preds.fillna(robust_rolling).fillna(rto_mean).fillna(0)
    return np.clip(fallback_preds, 0, None).to_dict()


def detect_anomalous_stores(df, threshold=100.0):
    """
    Обучение диагностической модели на периодах до февраля 2025.
    Оценка качества прогноза на феврале и выявление магазинов с APE > threshold.
    """
    feb_2025_time_idx = 2025 * 12 + 2

    # Минимальный набор лагов для оценки февраля
    for l in [1, 2, 3, 12, 13]:
        df[f'lag_{l}'] = df.groupby('new_id')['РТО'].shift(l)

    df['target_log_ratio'] = np.log((df['РТО'] + 1) / (df['lag_1'] + 1))
    df['snaive_pred'] = df['lag_12'] * (df['lag_1'] / (df['lag_13'] + 1e-6))
    df['snaive_pred'] = np.clip(df['snaive_pred'], 0, None)
    df['log_snaive_pred'] = np.log1p(df['snaive_pred'])
    df['snaive_log_ratio'] = np.log((df['snaive_pred'] + 1) / (df['lag_1'] + 1))
    df['yoy_growth'] = df['lag_1'] / (df['lag_13'] + 1e-6)
    df['mom_growth'] = df['lag_1'] / (df['lag_2'] + 1e-6)
    df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)

    categorical_features = ['Регион', 'Флаг алкогольной лицензии']
    for col in categorical_features:
        df[col] = df[col].astype(str).fillna('NA')

    features_light = [
        'Месяц', 'month_sin', 'month_cos', 'yoy_growth', 'mom_growth',
        'log_snaive_pred', 'snaive_log_ratio', 'lag_1', 'lag_2', 'lag_3'
    ] + categorical_features

    train_feb = df[(df['time_idx'] < feb_2025_time_idx) & df['target_log_ratio'].notna() & df['lag_13'].notna()].copy()
    val_feb = df[df['time_idx'] == feb_2025_time_idx].copy()
    train_feb['sample_weight'] = np.exp((train_feb['time_idx'] - train_feb['time_idx'].min()) / 12)

    unique_shops = train_feb['new_id'].unique()
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    feb_preds = np.zeros(len(val_feb))

    # Проверка доступности GPU для локального запуска
    try:
        import torch
        device_type = 'GPU' if torch.cuda.is_available() else 'CPU'
    except ImportError:
        device_type = 'CPU'

    print(f"Запуск диагностического обучения CatBoost (используется {device_type})...")

    for fold, (train_idx, val_idx) in enumerate(kf.split(unique_shops)):
        shops_train = unique_shops[train_idx]
        d_train = train_feb[train_feb['new_id'].isin(shops_train)].copy()
        
        # Сглаженный таргет-энкодинг регионов и населенных пунктов
        reg_map = d_train.groupby('Регион')['target_log_ratio'].mean().to_dict()
        city_map = d_train.groupby('Населенный пункт')['target_log_ratio'].mean().to_dict()
        
        d_train['reg_target_enc'] = d_train['Регион'].map(reg_map)
        d_train['city_target_enc'] = d_train['Населенный пункт'].map(city_map)
        
        val_feb_fold = val_feb.copy()
        val_feb_fold['reg_target_enc'] = val_feb_fold['Регион'].map(reg_map).fillna(d_train['target_log_ratio'].mean())
        val_feb_fold['city_target_enc'] = val_feb_fold['Населенный пункт'].map(city_map).fillna(d_train['target_log_ratio'].mean())
        
        curr_feats = features_light + ['reg_target_enc', 'city_target_enc']
        
        model_cv = CatBoostRegressor(
            iterations=500,
            learning_rate=0.03,
            depth=6,
            loss_function='MAE',
            random_seed=42 + fold,
            verbose=False,
            task_type=device_type
        )
        model_cv.fit(
            d_train[curr_feats], 
            d_train['target_log_ratio'], 
            sample_weight=d_train['sample_weight'], 
            cat_features=categorical_features
        )
        feb_preds += model_cv.predict(val_feb_fold[curr_feats]) / 5.0

    val_feb['pred_log_ratio'] = feb_preds
    val_feb['pred_rto'] = (val_feb['lag_1'].values + 1) * np.exp(val_feb['pred_log_ratio']) - 1
    val_feb['pred_rto'] = np.clip(val_feb['pred_rto'], 0, None)
    val_feb['APE'] = np.abs(val_feb['РТО'] - val_feb['pred_rto']) / (val_feb['РТО'] + 1e-6) * 100

    outlier_ids = set(val_feb[val_feb['APE'] > threshold]['new_id'].unique())
    return outlier_ids


def main():
    print("Шаг 1. Загрузка исторических данных...")
    try:
        df = load_history(TRAIN_PATH)
    except FileNotFoundError as e:
        print(f"Ошибка: {e}")
        return

    print("Шаг 2. Расчет робастных исторических трендов...")
    fallback_dict = calculate_fallback_predictions(df)

    print("Шаг 3. Поиск аномальных магазинов на февральском тесте...")
    outlier_ids = detect_anomalous_stores(df, threshold=APE_THRESHOLD)
    print(f"Обнаружено магазинов с повышенным уровнем ошибки (> {APE_THRESHOLD}%): {len(outlier_ids)}")

    print("Шаг 4. Чтение базового файла посылки...")
    if not os.path.exists(BASELINE_SUB_PATH):
        print(f"Ошибка: Не найден базовый файл прогноза '{BASELINE_SUB_PATH}'.")
        print("Сначала запустите pipeline.py, чтобы сгенерировать базовый прогноз.")
        return

    sub_df = pd.read_csv(BASELINE_SUB_PATH)

    print("Шаг 5. Применение коррекции к аномальным точкам...")
    replaced_counter = 0
    for idx, row in sub_df.iterrows():
        shop_id = row['new_id']
        if shop_id in outlier_ids:
            sub_df.at[idx, 'rto'] = fallback_dict.get(shop_id, row['rto'])
            replaced_counter += 1

    sub_df.to_csv(FINAL_SUB_PATH, index=False)
    print(f"Корректировка успешно завершена. Заменено значений: {replaced_counter} из {len(sub_df)}")
    print(f"Итоговый файл сохранен как '{FINAL_SUB_PATH}'")


if __name__ == '__main__':
    main()